In [1]:
import pandas as pd
import numpy as np

from functions import load_month, load_month_sa, load_month_pen
from rating_variables import rating_vars_2011

In [2]:
MONTHLIST = [f"{m:02d}" for m in range(1, 13)]
Y = 2011

In [3]:
plan_data = (
    pd.concat([load_month(m, Y) for m in MONTHLIST], ignore_index=True)
    .sort_values(["contractid", "planid", "state", "county", "month"])
)


plan_data["fips"] = (
    plan_data.groupby(["state", "county"])["fips"]
    .transform(lambda s: s.ffill().bfill())
)

for col in ["plan_type", "partd", "snp", "eghp", "plan_name"]:
    plan_data[col] = (
        plan_data.groupby(["contractid", "planid"])[col]
        .transform(lambda s: s.ffill().bfill())
    )

for col in ["org_type", "org_name", "org_marketing_name", "parent_org"]:
    plan_data[col] = (
        plan_data.groupby("contractid")[col]
        .transform(lambda s: s.ffill().bfill())
    )

plan_data = plan_data.sort_values(["contractid", "planid", "fips", "year", "month"])

/tmp/ipykernel_3966355/357241425.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda s: s.ffill().bfill())


In [4]:
def summarize_enrollment(g):
    enr = g["enrollment"].dropna()
    n = len(enr)
    return pd.Series({
        "n_nonmiss":          n,
        "avg_enrollment":     enr.mean()   if n > 0 else np.nan,
        "sd_enrollment":      enr.std()    if n > 1 else np.nan,
        "min_enrollment":     enr.min()    if n > 0 else np.nan,
        "max_enrollment":     enr.max()    if n > 0 else np.nan,
        "first_enrollment":   enr.iloc[0]  if n > 0 else np.nan,
        "last_enrollment":    enr.iloc[-1] if n > 0 else np.nan,
        "state":              g["state"].iloc[-1],
        "county":             g["county"].iloc[-1],
        "org_type":           g["org_type"].iloc[-1],
        "plan_type":          g["plan_type"].iloc[-1],
        "partd":              g["partd"].iloc[-1],
        "snp":                g["snp"].iloc[-1],
        "eghp":               g["eghp"].iloc[-1],
        "org_name":           g["org_name"].iloc[-1],
        "org_marketing_name": g["org_marketing_name"].iloc[-1],
        "plan_name":          g["plan_name"].iloc[-1],
        "parent_org":         g["parent_org"].iloc[-1],
        "contract_date":      g["contract_date"].iloc[-1],
    })

In [5]:
plan_data_year = (
    plan_data
    .groupby(["contractid", "planid", "fips", "year"], sort=False)
    .apply(summarize_enrollment)
    .reset_index()
)

/tmp/ipykernel_3966355/3073591061.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_enrollment)


In [6]:
service_year = (
    pd.concat([load_month_sa(m, Y) for m in MONTHLIST], ignore_index=True)
    .sort_values(["contractid", "fips", "state", "county", "month"])
)

service_year["fips"] = (
    service_year.groupby(["state", "county"])["fips"]
    .transform(lambda s: s.ffill().bfill())
)

for col in ["plan_type", "partial", "eghp", "org_type", "org_name"]:
    service_year[col] = (
        service_year.groupby("contractid")[col]
        .transform(lambda s: s.ffill().bfill())
    )

service_data_year = (
    service_year
    .sort_values(["contractid", "fips", "year", "month"])
    .groupby(["contractid", "fips", "year"], sort=False)
    .agg(
        state     = ("state",     "last"),
        county    = ("county",    "last"),
        org_name  = ("org_name",  "last"),
        org_type  = ("org_type",  "last"),
        plan_type = ("plan_type", "last"),
        partial   = ("partial",   "last"),
        eghp      = ("eghp",      "last"),
        ssa       = ("ssa",       "last"),
        notes     = ("notes",     "last"),
    )
    .reset_index()
)

/tmp/ipykernel_3966355/6740371.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda s: s.ffill().bfill())
/tmp/ipykernel_3966355/6740371.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda s: s.ffill().bfill())


In [7]:
ma_penetration = (
    pd.concat([load_month_pen(m, Y) for m in MONTHLIST], ignore_index=True)
    .sort_values(["state", "county", "month"])
)

ma_penetration["fips"] = (
    ma_penetration.groupby(["state", "county"])["fips"]
    .transform(lambda s: s.ffill().bfill())
)


def summarize_penetration(g):
    elig  = g["eligibles"].dropna()
    enrol = g["enrolled"].dropna()
    n_e, n_n = len(elig), len(enrol)
    return pd.Series({
        "n_elig":          n_e,
        "n_enrol":         n_n,
        "avg_eligibles":   elig.mean()   if n_e > 0 else np.nan,
        "sd_eligibles":    elig.std()    if n_e > 1 else np.nan,
        "min_eligibles":   elig.min()    if n_e > 0 else np.nan,
        "max_eligibles":   elig.max()    if n_e > 0 else np.nan,
        "first_eligibles": elig.iloc[0]  if n_e > 0 else np.nan,
        "last_eligibles":  elig.iloc[-1] if n_e > 0 else np.nan,
        "avg_enrolled":    enrol.mean()   if n_n > 0 else np.nan,
        "sd_enrolled":     enrol.std()    if n_n > 1 else np.nan,
        "min_enrolled":    enrol.min()    if n_n > 0 else np.nan,
        "max_enrolled":    enrol.max()    if n_n > 0 else np.nan,
        "first_enrolled":  enrol.iloc[0]  if n_n > 0 else np.nan,
        "last_enrolled":   enrol.iloc[-1] if n_n > 0 else np.nan,
        "ssa":             g["ssa"].iloc[-1],
    })


pen_year = (
    ma_penetration
    .sort_values(["fips", "state", "county", "year", "month"])
    .groupby(["fips", "state", "county", "year"], sort=False)
    .apply(summarize_penetration)
    .reset_index()
)

/tmp/ipykernel_3966355/510917193.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_penetration)


In [21]:
MA_PATH_A = (
    "../../ma-data/ma/star-ratings/Extracted Star Ratings/2011/"
    "2011_Part_C_Report_Card_Master_Table_2011_04_20_domain.csv"
)

MA_PATH_B = (
    "../../ma-data/ma/star-ratings/Extracted Star Ratings/2011/"
    "2011_Part_C_Report_Card_Master_Table_2011_04_20_summary.csv"
)
star_data_a = pd.read_csv(
    MA_PATH_A, skiprows=2, header=None,
    names=["contractid", "org_type", "contract_name", "org_marketing",
           "domain1", "domain2", "domain3", "domain4", "domain5"],
    na_values=["", "NA", "*", "Plan not required to report measure",
               "Not enough data to calculate stars for this category",
               "Not available", "Plan too new to be measured"],
    encoding='latin1',
)

id_cols = ["contractid", "org_type", "contract_name", "org_marketing"]
for col in ["domain1", "domain2", "domain3", "domain4", "domain5"]:
    star_data_a[col] = (
        star_data_a[col]
        .astype(str)
        .str.extract(r'(\d+\.?\d*)')[0]
        .astype(float)
    )

star_data_a["contractid"] = star_data_a["contractid"].astype(str)

star_data_b = pd.read_csv(
    MA_PATH_B, skiprows=2, header=None,
    names=["contractid", "org_type", "contract_name", "org_marketing",
           "low_performance", "partc_score", "overall_rating"],
    na_values=["", "NA", "*", "Not enough data to calculate summary rating",
               "Not enough data to calculate overall rating",
               "Not available"],  # removed "Plan too new to be measured"
    encoding='latin1',
)

star_data_b["contractid"] = star_data_b["contractid"].astype(str)

star_data_b["new_contract"] = (
    star_data_b["partc_score"] == "Plan too new to be measured"
).astype(int)

star_data_b["partc_score"] = np.where(
    star_data_b["new_contract"] == 1,
    np.nan,
    star_data_b["partc_score"].str.extract(r'(\d+\.?\d*)')[0].astype(float)
)

star_data_b = star_data_b[["contractid", "new_contract", "partc_score"]]
star_data_b["partcd_score"] = np.nan

star_data_year = (
    star_data_a
    .drop(columns=["contract_name", "org_type", "org_marketing","year",
                   "domain1", "domain2", "domain3", "domain4", "domain5"], errors="ignore")
    .merge(star_data_b, on="contractid", how="left")
    .assign(year=2011)
)



In [22]:
bench_data = pd.read_csv(
    "../../ma-data/ma/benchmarks/ratebook2011/CountyRate2011.csv",
    skiprows=10, header=None,
    names=["ssa", "state", "county_name", "aged_parta", "aged_partb",
           "disabled_parta", "disabled_partb", "esrd_ab", "risk_ab"],
)

bench_data_year = (
    bench_data[["ssa", "aged_parta", "aged_partb", "risk_ab"]]
    .assign(
        ssa          = lambda df: pd.to_numeric(df["ssa"], errors="coerce"),
        risk_star5   = np.nan, risk_star45  = np.nan, risk_star4   = np.nan,
        risk_star35  = np.nan, risk_star3   = np.nan, risk_star25  = np.nan,
        risk_bonus5  = np.nan, risk_bonus35 = np.nan, risk_bonus0  = np.nan,
        year         = 2011,
    )
)

In [23]:
ma_data = (
    plan_data_year
    .merge(
        service_data_year[["contractid", "fips"]],
        on=["contractid", "fips"],
        how="inner",
    )
    .query(
        "state not in ['VI','PR','MP','GU','AS',''] "
        "and snp == 'No' "
        "and (planid < 800 or planid >= 900) "
        "and planid.notna() "
        "and fips.notna()"
    )
)

# Prepare pen_year for merge (keep only fips that appear exactly once)
pen_merge = (
    pen_year
    .rename(columns={"state": "state_long", "county": "county_long"})
    .assign(state_long=lambda df: df["state_long"].str.lower())
)
pen_merge["ncount"] = pen_merge.groupby("fips")["fips"].transform("count")
pen_merge = pen_merge[pen_merge["ncount"] == 1].drop(columns="ncount")

ma_data = ma_data.merge(pen_merge, on="fips", how="left")

# Merge star ratings (drop duplicate org cols if present)
star_cols_to_drop = [c for c in ["contract_name", "org_type", "org_marketing"]
                     if c in star_data_year.columns]
ma_data = ma_data.merge(
    star_data_year.drop(columns=star_cols_to_drop, errors="ignore"),
    on="contractid", how="left",
)

# Compute Star_Rating
conditions = [
    ma_data["partd"] == "No",
    (ma_data["partd"] == "Yes") & ma_data["partcd_score"].isna(),
    (ma_data["partd"] == "Yes") & ma_data["partcd_score"].notna(),
]
choices = [
    ma_data["partc_score"],
    ma_data["partc_score"],
    ma_data["partcd_score"],
]
ma_data["Star_Rating"] = np.select(conditions, choices, default=np.nan)

# Merge benchmarks
ma_data = ma_data.merge(
    bench_data_year[bench_data_year["ssa"].notna()].drop(columns=["year"]),
    on="ssa", how="left",
)

# Drop duplicate year columns, set year = Y
year_cols = [c for c in ma_data.columns if c.startswith("year_")]
ma_data = ma_data.drop(columns=year_cols)
ma_data["year"] = Y

# Compute ma_rate
def ma_rate(row):
    yr = row["year"]
    sr = row["Star_Rating"]
    if yr < 2012:
        return row["risk_ab"]
    elif yr < 2015:
        if sr == 5:    return row["risk_star5"]
        if sr == 4.5:  return row["risk_star45"]
        if sr == 4:    return row["risk_star4"]
        if sr == 3.5:  return row["risk_star35"]
        if sr == 3:    return row["risk_star3"]
        if pd.notna(sr) and sr < 3: return row["risk_star25"]
        return row["risk_star35"]   # na Star_Rating
    else:  # yr >= 2015
        if pd.notna(sr) and sr >= 4: return row["risk_bonus5"]
        if pd.notna(sr) and sr < 4:  return row["risk_bonus0"]
        return row["risk_bonus35"]  # na Star_Rating

ma_data["ma_rate"] = ma_data.apply(ma_rate, axis=1)

In [25]:
ma_data.to_csv(f"../data/data-{Y}.csv", index=False)
print(f"Saved data-{Y}.csv with {len(ma_data):,} rows.")

Saved data-2011.csv with 68,003 rows.
